In [1]:
import polars as pl
from src.features import ICARELIST, build_features, ARCHIVELIST
from src.train import train_w_folds, TEST_END, TRAIN_END, report
from dateutil.relativedelta import relativedelta
from datetime import datetime, UTC

df = pl.read_parquet("../data/parquets/sold_listings_20260901.parquet").lazy()
def train_eval(df: pl.LazyDataFrame):
    df = build_features(df)
    df = df.with_columns(
        pl.col('primary_designer').is_in(ARCHIVELIST).alias('is_archive'),
        pl.col('primary_designer').is_in(ICARELIST).alias('brands_icare')
    )
    start = datetime(2026, 2, 1, tzinfo=UTC)
    start = start - relativedelta(months=12)
    end = datetime(2026, 3, 1, tzinfo=UTC)
    return train_w_folds(df, TRAIN_END, TEST_END)
folds = train_eval(df)
report(folds)

0.9131908503956614
(4241419, 60000) (4241419, 256) (57936, 60000) (57936, 256)


C:\Users\mononoaware\OneDrive\Documents\GitHub\GrailedDataAnalysis\.venv\Lib\site-packages\sklearn\utils\validation.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


[LightGBM] [Warning] Met categorical feature which contains sparse values. Consider renumbering to consecutive integers started from zero
[LightGBM] [Warning] Met categorical feature which contains sparse values. Consider renumbering to consecutive integers started from zero
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 160.990436 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 575058
[LightGBM] [Info] Number of data points in the train set: 4241419, number of used features: 59143
[LightGBM] [Info] Start training from score 4.418691
57936


[{'RMSE': 0.615936209074175,
  'MAE': 0.46778665439357114,
  'MAPE': 0.5475173273836285,
  'WITHIN_20%': 0.2917184479425573}]

In [9]:
print('All', report(folds))
print('Archive', report(folds, 'archivelist'))
print('Same titles', report(folds, 'same_titles'))
print('Unique_titles', report(folds, 'unique_titles'))
print('Unseen_titles', report(folds, 'unseen_titles'))
print('Within 50-250', report(folds, 'within_50_250'))

57936
All [{'RMSE': 0.615936209074175, 'MAE': 0.46778665439357114, 'MAPE': 0.5475173273836285, 'WITHIN_20%': 0.2917184479425573}]
3256
Archive [{'RMSE': 0.6156594635101836, 'MAE': 0.4702123958049466, 'MAPE': 0.5081981586068595, 'WITHIN_20%': 0.29484029484029484}]
2542
Same titles [{'RMSE': 0.5142071323374154, 'MAE': 0.3963449228968808, 'MAPE': 0.47089849643287407, 'WITHIN_20%': 0.34028324154209283}]
5237
Unique_titles [{'RMSE': 0.5900764254921342, 'MAE': 0.4498050183416004, 'MAPE': 0.5163933422874843, 'WITHIN_20%': 0.29940805804850107}]
43828
Unseen_titles [{'RMSE': 0.6312011200036498, 'MAE': 0.4800411525357244, 'MAPE': 0.5591079713479978, 'WITHIN_20%': 0.2833348544309574}]
Within 50-250 []
